In [10]:
from cryptography.hazmat.primitives.asymmetric import rsa

# Alice generate the key's
sk_a = rsa.generate_private_key(public_exponent=65537,
    key_size=4096)

pk_a = sk_a.public_key()

In [11]:
import os
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives import hashes
# Bob generates a secret key

sk_b = os.urandom(16)

# The Padding are shared
def get_padding():
    return padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )

# And encrypts it with Alices Public key
c = pk_a.encrypt(sk_b, get_padding())

In [12]:
# Alice decrypts the sk key

sk_a = sk_a.decrypt(c, get_padding())

In [13]:
assert sk_a == sk_b, "The secret key are not the same"

In [14]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
# Alice Encrypts a message with sk_a with AES

iv = os.urandom(16)
cipher_a = Cipher(algorithms.AES128(sk_a), modes.GCM(iv))

encryptor = cipher_a.encryptor()
c_aes = encryptor.update(b"Hello World") + encryptor.finalize()
tag = encryptor.tag

In [16]:
# Bob decrypts the message with sk_b, he received the iv and the tag and c_aes from Alice

cipher_b = Cipher(algorithms.AES128(sk_b), modes.GCM(iv, tag))
decryptor = cipher_b.decryptor()
m = decryptor.update(c_aes) + decryptor.finalize()

print(m)

b'Hello World'
